# Day 14：阶段考核项目一——命令行多轮对话 AI 助手（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-014` ｜ **开发分支**：`feat/day14-milestone1-cli-chat-assistant`  
> **配套 HTML5 课件**：`courseware/day14/index.html` ｜ **独立版本快照**：`versions/day14/`

---

## 🎙️ 老陈开场旁白

不知不觉来到第 14 天——**第一阶段毕业考核日（STARMIND-014）**！
林产品今天亲自来验收咱们的「星辰智脑 CLI 多轮对话助手 v1.0」。需求很明确，不再是 Day 12 那种问完就忘的单轮玩具，而是真正能连续聊天的生产力工具：
1）**多轮上下文记忆**：维护标准 `messages` 对话数组，并带滑动窗口裁剪，防止聊久了 Token 爆炸；
2）**JSON 对话持久化**：敲 `/save` 一键把整段对话保存到本地 JSON，下次还能 `load_from_json` 完整恢复接着聊；
3）**完整斜杠指令集**：`/help`、`/history`、`/stats`、`/clear`（清空历史但保留系统人设）、`/exit`；
4）**生产级异常处理**：后端报错不闪退，而且必须把刚才塞进去却没拿到回复的那条 `user` 消息**事务回滚**掉！
今天把这两份代码和全套 pytest 验收测试跑绿，你的第一阶段就圆满毕业了！

### 承上启下
- **昨日基线**：昨天咱们拿下了装饰器、`yield` 生成器、`python-dotenv` 和 `asyncio` 异步并发，第一阶段所有的核心技术拼图已经全部集齐。
- **明日预告**：恭喜通关第一阶段（14 天 Python 工程基础）！下周一（Day 15，第二阶段：Web 后端与数据库工程）咱们将把今天这个命令行助手正式升级成企业级 **FastAPI + 数据库持久化** 的多人在线 Web 服务！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day14'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-014）

作为星辰智脑第一阶段里程碑考核交付物，研发团队需要交付一个可独立运行、可测试、具备完整多轮对话体验的命令行 AI 助手 v1.0，作为后续 Web 版与 Agent 版的最小可行内核（MVP）。

**今日迭代目标：**
- 实现 `ConversationMemory` 类：管理 `system` 人设、`messages` 多轮历史、滑动窗口裁剪、Token 估算与 JSON 持久化读写。
- 实现 `StarMindCLIAssistant` 类：内置斜杠指令路由（`/help`、`/history`、`/stats`、`/clear`、`/save`、`/exit`）与可替换的对话后端接口。
- 实现异常事务回滚机制：模型调用失败时不崩溃退出，并自动撤销本次未完成的 `user` 消息。

**验收标准：**
- 连续多轮提问时，后端能接收到完整历史 `messages`（含首条 `system` 消息），且超出窗口上限时自动保留最新 N 条对话。
- `/clear` 清空对话历史但保留 `system_prompt`；`/save` 成功落盘 UTF-8 JSON 并可通过 `load_from_json` 完整还原。
- 模型后端抛出异常时，助手返回友好错误提示、不崩溃，且记忆中不会残留失败的 `user` 消息。
- 全部核心类与斜杠指令通过无阻塞 `pytest` 自动化验收测试。

---

## 📚 课堂核心技术笔记

### 1. 1. 多轮对话本质：大模型是「无状态」的

很多人误以为大模型服务器替你记住了上一句话，其实 HTTP API 是完全无状态（Stateless）的！所谓「多轮记忆」，全靠客户端把 `[system, user_1, assistant_1, user_2, assistant_2, user_3]` 整串历史记录每次重新打包发给模型。
因此客户端必须做好**滑动窗口裁剪**（始终保留 `system` + 最近 N 条消息），否则聊上 50 轮不仅 Token 费暴涨，还会直接触发模型上下文长度超限报错。

> **⚠️ 老陈避坑**：裁剪历史消息时，千万别一刀把索引 0 的 `system` 人设消息也截掉了！务必把 `system_prompt` 独立保存，每次组装 payload 时再拼在最前面。

### 2. 2. 失败回滚：保护 messages 奇偶交替完整性

主流大模型 API 严格要求 `messages` 中 `user` 和 `assistant` 交替出现。如果在调用模型前先执行了 `add_user_message()`，结果这次网络超时报错了，如果不把刚才那条 `user` 消息 `pop` 回滚掉，下一次用户重新提问就会出现连续两条 `user` 消息，导致后续所有请求被 API 拒绝（HTTP 400）！

> **⚠️ 老陈避坑**：在 `except Exception:` 分支里调用 `memory.rollback_last_user_message()`，是一个极小但极显功力的生产级细节。

### 3. 3. CLI 项目的工程化可测试性（依赖注入）

写命令行程序最容易犯的错，就是把 `while True: input()`、文件读写和网络请求全揉在一个文件顶层，导致完全没法写单元测试。
正确做法是：**将单行处理逻辑抽成纯方法 `handle_line(raw_line) -> (should_continue, reply)`**，并把后端引擎 `backend` 和保存目录 `save_dir` 通过构造函数注入——这样 `pytest` 不用模拟键盘敲击也能 100% 测完每一条斜杠指令！

> **⚠️ 老陈避坑**：涉及文件名参数的 `/save <filename>` 指令，要注意用 `Path(filename).name` 去掉恶意路径前缀（如 `../../etc/passwd`），防止路径穿越漏洞。


## 💻 实操步骤 1：步骤一：实现多轮对话记忆引擎与 JSON 持久化模块

- **目标文件**：`versions/day14/src/day14_session_memory.py`
- **老陈代码旁白**：先搭好核心数据结构 `ConversationMemory`：负责 `messages` 增删、滑动窗口截断、失败回滚、Token 估算以及 UTF-8 JSON 持久化存取。


In [ ]:
"""
STARMIND-014 模块一：多轮对话记忆引擎与 JSON 持久化存储。
负责维护 system 人设、多轮 messages 历史、滑动窗口裁剪、异常回滚与 JSON 读写。
"""

import json
import tempfile
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List


@dataclass
class ChatMessage:
    """单条对话消息记录。"""

    role: str  # "user" 或 "assistant"
    content: str
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat(timespec="seconds")
    )


class ConversationMemory:
    """星辰智脑多轮对话上下文记忆管理器。"""

    def __init__(
        self,
        system_prompt: str = "你是杭州星辰数科企业 AI 助手「星辰智脑」，请提供专业简洁的中文解答。",
        max_history_messages: int = 10,
        session_id: str = "default-session",
    ) -> None:
        if max_history_messages < 2:
            raise ValueError("max_history_messages 至少为 2（一问一答）")
        self.system_prompt = system_prompt
        self.max_history_messages = max_history_messages
        self.session_id = session_id
        self.messages: List[ChatMessage] = []

    def add_user_message(self, content: str) -> ChatMessage:
        """追加一条用户消息。"""
        cleaned = content.strip()
        if not cleaned:
            raise ValueError("用户消息不能为空")
        msg = ChatMessage(role="user", content=cleaned)
        self.messages.append(msg)
        return msg

    def add_assistant_message(self, content: str) -> ChatMessage:
        """追加一条助手回复消息。"""
        msg = ChatMessage(role="assistant", content=content.strip())
        self.messages.append(msg)
        return msg

    def rollback_last_user_message(self) -> bool:
        """当模型调用失败时回滚最后一条孤立的 user 消息，防止上下文损坏。"""
        if self.messages and self.messages[-1].role == "user":
            self.messages.pop()
            return True
        return False

    def build_payload_messages(self) -> List[Dict[str, str]]:
        """
        构造发送给大模型的标准 messages 数组：
        始终保留 system_prompt 在首位 + 最近 max_history_messages 条历史消息。
        """
        window = self.messages[-self.max_history_messages :]
        payload: List[Dict[str, str]] = [{"role": "system", "content": self.system_prompt}]
        for item in window:
            payload.append({"role": item.role, "content": item.content})
        return payload

    def clear(self) -> None:
        """清空多轮历史消息（保留 system_prompt 人设）。"""
        self.messages.clear()

    def get_stats(self) -> Dict[str, Any]:
        """统计当前会话轮数、消息总数及粗略 Token 估算。"""
        user_turns = sum(1 for m in self.messages if m.role == "user")
        total_chars = len(self.system_prompt) + sum(len(m.content) for m in self.messages)
        return {
            "session_id": self.session_id,
            "turns": user_turns,
            "message_count": len(self.messages),
            "estimated_tokens": max(1, int(total_chars * 1.2)),
            "window_limit": self.max_history_messages,
        }

    def save_to_json(self, file_path: Path) -> Path:
        """将当前多轮对话完整持久化为 UTF-8 JSON 文件。"""
        file_path.parent.mkdir(parents=True, exist_ok=True)
        data = {
            "session_id": self.session_id,
            "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "system_prompt": self.system_prompt,
            "max_history_messages": self.max_history_messages,
            "messages": [asdict(m) for m in self.messages],
        }
        file_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        return file_path

    @classmethod
    def load_from_json(cls, file_path: Path) -> "ConversationMemory":
        """从 JSON 存档文件恢复历史会话记忆。"""
        if not file_path.exists():
            raise FileNotFoundError(f"会话文件不存在: {file_path}")
        try:
            raw = json.loads(file_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            raise ValueError(f"会话 JSON 文件已损坏: {exc}") from exc

        if not isinstance(raw, dict) or not isinstance(raw.get("messages"), list):
            raise ValueError("会话 JSON 结构无效：缺少 messages 列表")

        mem = cls(
            system_prompt=str(raw.get("system_prompt", "")),
            max_history_messages=int(raw.get("max_history_messages", 10)),
            session_id=str(raw.get("session_id", "restored-session")),
        )
        for item in raw["messages"]:
            mem.messages.append(
                ChatMessage(
                    role=str(item["role"]),
                    content=str(item["content"]),
                    timestamp=str(item.get("timestamp", "")),
                )
            )
        return mem


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        mem = ConversationMemory(session_id="demo-014", max_history_messages=4)
        mem.add_user_message("我叫小王，负责杭州研发部。")
        mem.add_assistant_message("你好小王！很高兴为杭州研发部服务。")
        saved = mem.save_to_json(Path(tmp) / "chat_history.json")
        restored = ConversationMemory.load_from_json(saved)
        print("持久化恢复成功，统计信息:", restored.get_stats())
        print("Payload 消息数:", len(restored.build_payload_messages()))


### 🔍 步骤 1 代码深度解析

1. `build_payload_messages()` 通过切片 `self.messages[-self.max_history_messages:]` 实现优雅的滑动窗口裁剪，同时确保 `system_prompt` 永远稳居第一位。
2. `load_from_json()` 是 `@classmethod` 工厂方法，负责校验文件存在性、JSON 合法性与字段完整性，读到损坏文件时抛出清晰的 `ValueError`。

**预期运行输出参考：**
```text
持久化恢复成功，统计信息: {'session_id': 'demo-014', 'turns': 1, 'message_count': 2, 'estimated_tokens': 81, 'window_limit': 4}
Payload 消息数: 3
```


## 💻 实操步骤 2：步骤二：实现支持斜杠指令与异常回滚的命令行多轮 AI 助手 v1.0

- **目标文件**：`versions/day14/src/day14_cli_assistant.py`
- **老陈代码旁白**：现在把记忆引擎、可插拔后端和斜杠指令系统拼装成完整的 `StarMindCLIAssistant`！注意看 `LocalSmartMockBackend` 会真正读取前几轮 `messages` 里的上下文（比如记住用户的名字），离线就能体验真实多轮记忆！


In [ ]:
"""
STARMIND-014 模块二：星辰智脑命令行多轮对话 AI 助手 v1.0。
支持多轮上下文记忆、/help /history /stats /clear /save /exit 斜杠指令、
JSON 持久化存档与模型异常自动回滚。
"""

import re
import tempfile
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from day14_session_memory import ConversationMemory


class BaseChatBackend(ABC):
    """大模型对话后端抽象基类，便于无缝切换本地 Mock 或 DeepSeek/通义千问 HTTP 客户端。"""

    @abstractmethod
    def complete(self, messages: List[Dict[str, str]]) -> str:
        """接收标准 OpenAI 格式 messages 列表并返回助手回复正文。"""


class LocalSmartMockBackend(BaseChatBackend):
    """具备基础多轮上下文记忆能力的离线模拟后端（零外网依赖）。"""

    NAME_PATTERN = re.compile(r"(?:我叫|我是)([\u4e00-\u9fa5a-zA-Z0-9_]{1,12})")

    def complete(self, messages: List[Dict[str, str]]) -> str:
        user_msgs = [m["content"] for m in messages if m.get("role") == "user"]
        if not user_msgs:
            return "你好！我是星辰智脑，请问有什么可以帮您？"

        latest = user_msgs[-1]
        if "__TRIGGER_ERROR__" in latest:
            raise RuntimeError("模拟上游大模型网关 503 临时故障")

        # 扫描全部历史 user 消息，提取用户此前介绍过的名字（演示多轮记忆）
        remembered_name: Optional[str] = None
        for text in user_msgs:
            match = self.NAME_PATTERN.search(text)
            if match:
                candidate = match.group(1)
                if not any(q in candidate for q in ("什么", "谁", "吗", "呢")):
                    remembered_name = candidate

        if "我是谁" in latest or "我叫什么" in latest or "记得我" in latest:
            if remembered_name:
                return f"当然记得！你之前告诉我，你叫「{remembered_name}」。当前已累计记忆 {len(user_msgs)} 轮提问。"
            return "你还没有告诉我你的名字呢，可以说「我叫XXX」让我记住你！"

        if "报销" in latest:
            prefix = f"{remembered_name}同学，" if remembered_name else ""
            return f"{prefix}星辰数科报销请在钉钉提交费用单，500元以上需主管与财务双审批。"

        prefix = f"收到，{remembered_name}！" if remembered_name else "收到！"
        return f"{prefix}你刚才说：「{latest}」（当前上下文共携带 {len(messages)} 条消息）。"


class StarMindCLIAssistant:
    """星辰智脑命令行多轮对话助手主控类。"""

    HELP_TEXT = (
        "可用指令列表：\n"
        "  /help              查看帮助信息\n"
        "  /history           查看当前对话历史\n"
        "  /stats             查看对话轮数与 Token 估算\n"
        "  /clear             清空当前多轮记忆（保留系统人设）\n"
        "  /save [文件名]     保存对话记录为 JSON 文件（默认 session_history.json）\n"
        "  /exit              退出助手"
    )

    def __init__(
        self,
        backend: Optional[BaseChatBackend] = None,
        memory: Optional[ConversationMemory] = None,
        save_dir: Optional[Path] = None,
    ) -> None:
        self.backend = backend or LocalSmartMockBackend()
        self.memory = memory or ConversationMemory()
        self.save_dir = save_dir or Path("sessions")

    def _handle_command(self, line: str) -> Tuple[bool, str]:
        """解析并执行以 / 开头的斜杠管理指令。"""
        parts = line.split(maxsplit=1)
        cmd = parts[0].lower()
        arg = parts[1].strip() if len(parts) > 1 else ""

        if cmd in {"/exit", "/quit"}:
            return False, "再见！感谢使用星辰智脑命令行 AI 助手。"

        if cmd == "/help":
            return True, self.HELP_TEXT

        if cmd == "/clear":
            self.memory.clear()
            return True, "已清空当前多轮对话记忆（系统人设已保留）。"

        if cmd == "/stats":
            st = self.memory.get_stats()
            return True, (
                f"会话ID: {st['session_id']} | 对话轮数: {st['turns']} | "
                f"消息条数: {st['message_count']} | 估算Tokens: {st['estimated_tokens']}"
            )

        if cmd == "/history":
            if not self.memory.messages:
                return True, "当前暂无对话历史。"
            lines = [f"[{i}] {m.role}: {m.content}" for i, m in enumerate(self.memory.messages, 1)]
            return True, "\n".join(lines)

        if cmd == "/save":
            # 使用 Path(arg).name 防止路径穿越漏洞
            safe_name = Path(arg).name if arg else "session_history.json"
            if not safe_name.endswith(".json"):
                safe_name += ".json"
            target = self.save_dir / safe_name
            saved_path = self.memory.save_to_json(target)
            return True, f"对话历史已保存至：{saved_path.as_posix()}"

        return True, f"未知指令：{cmd}，输入 /help 查看所有支持的指令。"

    def handle_line(self, raw_line: str) -> Tuple[bool, str]:
        """
        处理单行用户输入，返回 (是否继续运行, 输出文本)。
        """
        text = raw_line.strip()
        if not text:
            return True, "提示：输入不能为空，请输入问题或 /help 查看指令。"

        if text.startswith("/"):
            return self._handle_command(text)

        self.memory.add_user_message(text)
        try:
            payload = self.memory.build_payload_messages()
            reply = self.backend.complete(payload)
        except Exception as exc:
            # 异常时回滚刚才塞进去的 user 消息，确保历史状态不被污染
            self.memory.rollback_last_user_message()
            return True, f"抱歉，模型调用出现异常（已自动回滚本轮输入）：{exc}"

        self.memory.add_assistant_message(reply)
        return True, f"星辰智脑 > {reply}"

    def run_repl(self, input_lines: Optional[List[str]] = None) -> List[str]:
        """
        运行多轮对话主循环。
        - 传入 input_lines 时非阻塞逐行回放（用于演示与自动化测试）；
        - 未传 input_lines 时进入终端交互式 input() 模式。
        """
        transcript: List[str] = []
        script_iter = iter(input_lines) if input_lines is not None else None

        while True:
            if script_iter is not None:
                try:
                    raw = next(script_iter)
                except StopIteration:
                    break
            else:
                try:
                    raw = input("你 > ")
                except (EOFError, KeyboardInterrupt):
                    transcript.append("检测到中断信号，已安全退出。")
                    break

            keep_going, out = self.handle_line(raw)
            transcript.append(out)
            if not keep_going:
                break

        return transcript


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        assistant = StarMindCLIAssistant(save_dir=Path(tmp))
        demo_script = [
            "你好，我叫林产品，在杭州研发中心工作。",
            "你还记得我叫什么吗？",
            "/stats",
            "/save milestone1_demo.json",
            "/clear",
            "你还记得我叫什么吗？",
            "/exit",
        ]
        for user_in, bot_out in zip(demo_script, assistant.run_repl(demo_script)):
            print(f"你 > {user_in}")
            print(bot_out)
            print("-" * 55)


### 🔍 步骤 2 代码深度解析

1. `LocalSmartMockBackend` 通过正则扫描历史 `user` 消息来记住用户名字，当用户输入 `/clear` 后再追问「还记得我叫什么吗？」，助手会诚实回答不记得，直观验证了多轮记忆与清空指令的真实效果！
2. `/save` 指令使用 `Path(arg).name` 清洗文件名，杜绝用户误输 `../../xx.json` 覆盖系统其他目录文件。
3. `handle_line()` 在 `except Exception` 中执行 `rollback_last_user_message()`，彻底杜绝半截对话污染上下文。

**预期运行输出参考：**
```text
你 > 你好，我叫林产品，在杭州研发中心工作。
星辰智脑 > 收到，林产品！你刚才说：「你好，我叫林产品，在杭州研发中心工作。」（当前上下文共携带 2 条消息）。
-------------------------------------------------------
你 > 你还记得我叫什么吗？
星辰智脑 > 当然记得！你之前告诉我，你叫「林产品」。当前已累计记忆 2 轮提问。
-------------------------------------------------------
你 > /stats
会话ID: default-session | 对话轮数: 2 | 消息条数: 4 | 估算Tokens: 210
-------------------------------------------------------
你 > /save milestone1_demo.json
对话历史已保存至：.../milestone1_demo.json
-------------------------------------------------------
你 > /clear
已清空当前多轮对话记忆（系统人设已保留）。
-------------------------------------------------------
你 > 你还记得我叫什么吗？
星辰智脑 > 你还没有告诉我你的名字呢，可以说「我叫XXX」让我记住你！
-------------------------------------------------------
你 > /exit
再见！感谢使用星辰智脑命令行 AI 助手。
-------------------------------------------------------
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day14/tests/test_day14.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-014 阶段考核项目一自动化验收测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day14_session_memory import ConversationMemory
from day14_cli_assistant import StarMindCLIAssistant


def test_memory_sliding_window_and_json_roundtrip(tmp_path: Path):
    mem = ConversationMemory(system_prompt="你是星辰助教", max_history_messages=2, session_id="s-01")
    mem.add_user_message("第1问")
    mem.add_assistant_message("第1答")
    mem.add_user_message("第2问")
    mem.add_assistant_message("第2答")

    payload = mem.build_payload_messages()
    # system 置顶 + 最近 2 条消息 = 共 3 条
    assert len(payload) == 3
    assert payload[0] == {"role": "system", "content": "你是星辰助教"}
    assert payload[1]["content"] == "第2问"
    assert payload[2]["content"] == "第2答"

    saved_file = mem.save_to_json(tmp_path / "history.json")
    restored = ConversationMemory.load_from_json(saved_file)
    assert restored.session_id == "s-01"
    assert restored.system_prompt == "你是星辰助教"
    assert len(restored.messages) == 4

    # 损坏的 JSON 必须抛出 ValueError
    broken = tmp_path / "broken.json"
    broken.write_text("{not valid json", encoding="utf-8")
    with pytest.raises(ValueError):
        ConversationMemory.load_from_json(broken)


def test_cli_assistant_multiturn_commands_and_persistence(tmp_path: Path):
    assistant = StarMindCLIAssistant(save_dir=tmp_path)

    # 1. 多轮记忆验证
    ok1, out1 = assistant.handle_line("你好，我叫阿杰")
    assert ok1 is True and "阿杰" in out1

    ok2, out2 = assistant.handle_line("你还记得我叫什么吗？")
    assert ok2 is True and "阿杰" in out2

    # 2. /stats 与 /history 与 /save
    _, stats_out = assistant.handle_line("/stats")
    assert "对话轮数: 2" in stats_out

    _, hist_out = assistant.handle_line("/history")
    assert "阿杰" in hist_out

    _, save_out = assistant.handle_line("/save my_chat")
    saved_path = tmp_path / "my_chat.json"
    assert saved_path.exists()
    assert "my_chat.json" in save_out

    # 3. /clear 清空后忘记上下文，但系统人设仍在
    _, clear_out = assistant.handle_line("/clear")
    assert "已清空" in clear_out
    assert len(assistant.memory.messages) == 0

    _, out3 = assistant.handle_line("你还记得我叫什么吗？")
    assert "还没有告诉我" in out3

    # 4. /exit 正常结束循环
    keep_going, exit_out = assistant.handle_line("/exit")
    assert keep_going is False
    assert "再见" in exit_out


def test_cli_assistant_error_rollback_and_edge_cases(tmp_path: Path):
    assistant = StarMindCLIAssistant(save_dir=tmp_path)
    assistant.handle_line("第一轮正常提问")
    assert len(assistant.memory.messages) == 2

    # 触发模拟后端故障：助手不应崩溃，且失败的 user 消息应被自动回滚
    keep_going, err_out = assistant.handle_line("请执行 __TRIGGER_ERROR__")
    assert keep_going is True
    assert "已自动回滚" in err_out
    assert len(assistant.memory.messages) == 2

    # 空输入与未知指令
    _, empty_out = assistant.handle_line("   ")
    assert "不能为空" in empty_out

    _, unknown_out = assistant.handle_line("/foobar")
    assert "未知指令" in unknown_out


def test_run_repl_end_to_end(tmp_path: Path):
    assistant = StarMindCLIAssistant(save_dir=tmp_path)
    outputs = assistant.run_repl(["/help", "我叫老陈", "报销怎么走？", "/exit"])
    assert len(outputs) == 4
    assert "/clear" in outputs[0]
    assert "老陈同学" in outputs[2]
    assert "再见" in outputs[3]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为多轮对话增加 Markdown 导出器

**题目要求：**
编写函数 `export_messages_to_markdown(messages: list[dict], title: str = '星辰智脑对话记录') -> str`，将 `messages` 列表渲染为清晰的 Markdown 文本（每条消息以 `### 用户` 或 `### 助手` 标题分段）。

**思路提示：** 遍历 `messages`，用字典把 `'user'` 映射为 `'用户'`、`'assistant'` 映射为 `'助手'`，最后用 `'\n\n'.join(blocks)` 拼接。


In [ ]:
"""Day 14 课后题 1：多轮对话记录 Markdown 导出器。"""

from typing import Dict, List


def export_messages_to_markdown(
    messages: List[Dict[str, str]],
    title: str = "星辰智脑对话记录",
) -> str:
    """将标准 messages 列表导出为易读的 Markdown 纪要文本。"""
    role_map = {"user": "用户", "assistant": "星辰智脑", "system": "系统人设"}
    blocks = [f"# {title}"]
    for idx, msg in enumerate(messages, 1):
        role_label = role_map.get(msg.get("role", ""), msg.get("role", "未知"))
        blocks.append(f"### {idx}. {role_label}\n\n{msg.get('content', '').strip()}")
    return "\n\n".join(blocks)


if __name__ == "__main__":
    sample = [
        {"role": "user", "content": "星辰智脑有哪些指令？"},
        {"role": "assistant", "content": "支持 /help、/clear、/save、/stats 和 /exit。"},
    ]
    print(export_messages_to_markdown(sample))


**答案解析与评分标准：**

JSON 适合程序加载恢复，Markdown 适合直接粘贴到飞书/钉钉文档分享给同事，两者结合能极大提升 CLI 工具的实用性。


### 作业 2（企业实战）：企业实战：本地多会话存档管理器

**题目要求：**
编写 `list_saved_sessions(save_dir: Path) -> list[dict]`，扫描目录下所有 `.json` 会话存档，返回按文件名排序的摘要列表，每项包含 `file_name`、`session_id`、`message_count`；若遇到损坏的 JSON 文件不崩溃，将其 `session_id` 标记为 `'CORRUPTED'`。

**思路提示：** 用 `sorted(save_dir.glob('*.json'))` 遍历，配合 `try/except (json.JSONDecodeError, OSError)` 容错解析。


In [ ]:
"""Day 14 课后题 2：多会话 JSON 存档扫描与容错摘要器。"""

import json
import tempfile
from pathlib import Path
from typing import Any, Dict, List


def list_saved_sessions(save_dir: Path) -> List[Dict[str, Any]]:
    """扫描会话目录，返回所有 JSON 存档的摘要信息（损坏文件安全标记不崩溃）。"""
    if not save_dir.exists():
        return []
    summaries: List[Dict[str, Any]] = []
    for path in sorted(save_dir.glob("*.json")):
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            summaries.append({
                "file_name": path.name,
                "session_id": str(data.get("session_id", "unknown")),
                "message_count": len(data.get("messages", [])),
            })
        except (json.JSONDecodeError, OSError, TypeError):
            summaries.append({
                "file_name": path.name,
                "session_id": "CORRUPTED",
                "message_count": 0,
            })
    return summaries


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        (root / "s1.json").write_text(json.dumps({"session_id": "hr-qa", "messages": [{"role": "user", "content": "hi"}]}), encoding="utf-8")
        (root / "bad.json").write_text("{broken", encoding="utf-8")
        print(list_saved_sessions(root))


**答案解析与评分标准：**

目录扫描类功能一定要做单文件异常隔离：绝不能因为目录里混进一个半截损坏文件就导致整个 `/list` 列表功能崩溃。


### 作业 3（架构进阶）：架构进阶：基于 Token 预算的智能上下文裁剪器

**题目要求：**
按条数裁剪有时不够精准（一条长日志就可能占几千字符）。编写 `trim_messages_by_budget(system_prompt: str, messages: list[dict], max_chars: int) -> list[dict]`，始终保留首条 `system` 消息，然后从最新消息往前倒序装入，直到总字符数即将超过 `max_chars` 为止，最终保持时间正序返回。

**思路提示：** 先扣掉 `len(system_prompt)` 得到剩余预算，再用 `reversed(messages)` 倒序累加挑出能放下的最新消息，最后反转回来拼在 `system` 后面。


In [ ]:
"""Day 14 课后题 3：按字符/Token 预算倒序裁剪多轮上下文。"""

from typing import Dict, List


def trim_messages_by_budget(
    system_prompt: str,
    messages: List[Dict[str, str]],
    max_chars: int,
) -> List[Dict[str, str]]:
    """在严格不超过 max_chars 预算的前提下，保留 system 人设与尽可能多的新近消息。"""
    budget = max_chars - len(system_prompt)
    selected_rev: List[Dict[str, str]] = []

    for msg in reversed(messages):
        cost = len(msg.get("content", ""))
        if cost <= budget:
            selected_rev.append(msg)
            budget -= cost
        else:
            break

    selected_rev.reverse()
    return [{"role": "system", "content": system_prompt}, *selected_rev]


if __name__ == "__main__":
    history = [
        {"role": "user", "content": "这是一条很老很长的历史问题" * 3},
        {"role": "assistant", "content": "旧回复"},
        {"role": "user", "content": "最新问题"},
        {"role": "assistant", "content": "最新回复"},
    ]
    trimmed = trim_messages_by_budget("系统人设", history, max_chars=25)
    print(trimmed)


**答案解析与评分标准：**

倒序装包 + 反转还原是几乎所有主流 AI 框架（LangChain / LlamaIndex / OpenAI Agent）计算 Token 上下文窗口的底层标准算法！


---

## 🎯 老陈收尾复盘

啪啪啪！老陈给你鼓个掌——从 Day 1 连虚拟环境都不熟，到 Day 14 独立交付一个架构清晰、记忆可控、支持 JSON 持久化与异常回滚的多轮对话 CLI 助手，两周时间你已经跨过了 80% 自学者倒下的工程门槛！
周末好好复盘一下 `ConversationMemory` 和 `StarMindCLIAssistant` 的解耦设计：正是因为今天把「记忆存储」「模型后端」「交互外壳」拆得干干净净，下周咱们迁移到 FastAPI Web 接口时，核心业务代码几乎一行不用重写！
